In [1]:
import pandas as pd
import gc

/home/morrowto/.local/lib/python3.10/site-packages/numpy/_core/getlimits.py:551: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)


In [2]:
df = pd.read_csv('../../data/fines.csv')

In [3]:
%%timeit
df['calculated'] = df['Fines'] / df['Refund'] * df['Year']

143 μs ± 487 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [4]:
%%timeit
def calc_using_for(df):
    result = []
    for i in range(0, len(df)):
        fines = df.iloc[i]['Fines']
        refund = df.iloc[i]['Refund']
        year = df.iloc[i]['Year']
        if refund != 0:
            result.append(fines / refund * year)
        else:
            result.append('inf')
    return result

df['calculated_for'] = calc_using_for(df)

76.9 ms ± 1.38 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [5]:
%%timeit
def calc_using_iterrows(df):
    result = []
    for _, row in df.iterrows():
        fines = row['Fines']
        refund = row['Refund']
        year = row['Year']
        
        if refund != 0:
            result.append(fines / refund * year)
        else:
            result.append('inf')
    return result

df['calculated_iterrows'] = calc_using_iterrows(df)


26.1 ms ± 322 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [6]:
%%timeit
df['calculated_apply'] = df.apply(
    lambda row: row['Fines'] / row['Refund'] * row['Year'] if row['Refund'] != 0 else 'inf', axis=1
)

7.21 ms ± 40.4 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [7]:
%%timeit
df['calculated_series'] = pd.Series(df['Fines'].replace({',': ''}, regex=True).astype(float) / df['Refund'] * df['Year'])

226 μs ± 1.32 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [8]:
%%timeit
df['calculated_values'] = [
    (fines / refund * year if refund != 0 else 'inf') 
    for fines, refund, year in zip(df['Fines'].values, df['Refund'].values, df['Year'].values)
]

390 μs ± 2.45 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [9]:
%%timeit
car_number = 'O136HO197RUS'
df_copy = df.copy()
row_before = df_copy[df_copy['CarNumber'] == car_number]
df_copy.set_index('CarNumber', inplace=True)
row_after = df_copy.loc[car_number]

589 μs ± 2.13 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
df.info(memory_usage='deep') 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CarNumber            930 non-null    object 
 1   Refund               930 non-null    int64  
 2   Fines                930 non-null    float64
 3   Make                 930 non-null    object 
 4   Model                919 non-null    object 
 5   Year                 930 non-null    int64  
 6   calculated           930 non-null    float64
 7   calculated_for       930 non-null    object 
 8   calculated_iterrows  930 non-null    object 
 9   calculated_apply     930 non-null    object 
 10  calculated_series    930 non-null    float64
 11  calculated_values    930 non-null    object 
dtypes: float64(3), int64(2), object(7)
memory usage: 342.0 KB


In [11]:
df_optimized = df.copy()
df_optimized['Fines'] = pd.to_numeric(df_optimized['Fines'], downcast='float')
df_optimized['Year'] = pd.to_numeric(df_optimized['Year'], downcast='integer')
df_optimized['Refund'] = pd.to_numeric(df_optimized['Refund'], downcast='integer')
df_optimized['calculated'] = pd.to_numeric(df_optimized['calculated'], downcast = 'float')
df_optimized['calculated_series'] = pd.to_numeric(df_optimized['calculated_series'], downcast = 'float')